# 04 — Training: baseline, imbalance, models, CV, tuning

This notebook **explains** the training story. The reproducible full run is:

```bash
python -m src.train
```

that writes `models/model.pkl`, `models/model_metadata.json`, and `reports/model_report.md`.

## Classification recap

Input features → model → probability → threshold → {0,1}.

## Logistic regression

```
features x
    ↓
weighted sum  z = w·x + b
    ↓
sigmoid  σ(z) = 1 / (1 + e^{-z})
    ↓
probability of fraud
    ↓
threshold (default 0.5, later tuned as policy)
    ↓
prediction
```

1. **What:** a linear classifier in probability space.
2. **Why:** strong, inspectable **baseline**. If logistic regression already separates well, trees must beat it on **PR-AUC**, not vibes.
3. **How:** maximum likelihood for Bernoulli labels.
4. **Here:** we compare unweighted vs `class_weight="balanced"`.

## Decision tree

Recursive splits that isolate purer regions of fraud vs legit.

## Random forest

```
dataset
    ↓
many trees on bootstrap samples / feature subsets
    ↓
each tree votes (or averages probabilities)
    ↓
final prediction
```

## Gradient boosting / XGBoost

Trees are added **sequentially** to correct residual errors. XGBoost is a regularized, efficient implementation. `scale_pos_weight` ≈ negatives/positives addresses imbalance without synthesizing rows.

## Class imbalance methods

- **Class weights:** penalize mistakes on fraud more. No synthetic data.
- **Undersampling:** throw away legit rows. Fast, but wastes majority information.
- **SMOTE:**

```
minority class
    ↓
find nearby minority neighbors
    ↓
create synthetic points on the joining lines
    ↓
more balanced *training* data
```

**Never SMOTE before train/test split:** synthetic neighbors can be built from points that belong in the test set, leaking holdout geometry.

SMOTE also belongs **inside** cross-validation folds, not on the whole train set before CV (otherwise validation folds are not independent of the sampler).


In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data_processing import dataset_overview, drop_duplicates_if_present, load_raw_dataset
from src.utils import RAW_DATASET_PATH

import pandas as pd
from pathlib import Path
reports = ROOT / "reports"
for name in ["imbalance_comparison.csv", "cv_model_comparison.csv", "test_metrics.csv"]:
    path = reports / name
    print("\n===", name, "===")
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("Run `python -m src.train` to generate this table.")


## Cross-validation: StratifiedKFold

1. **What:** rotate which slice of *training* data is the validation fold.
2. **Why ordinary K-Fold fails here:** a fold might contain almost no fraud; ROC/PR scores become noisy or undefined.
3. **What stratification does:** each fold keeps roughly the same fraud rate.
4. **Why CV is more reliable than one split:** you see mean ± std of PR-AUC before touching the test set.

Hyperparameter search in `src/train.py` uses `RandomizedSearchCV` with `scoring="average_precision"` (PR-AUC) on **train** only.

## Parameters we search (and why)

**Random Forest / trees**

- `n_estimators`: more trees → more stable, slower.
- `max_depth`: deeper → more interaction capture, more overfit risk.
- `min_samples_split` / `min_samples_leaf`: stop tiny fraudulent leaves that do not generalize.
- `max_features`: randomness that decorrelates trees.

**XGBoost**

- `learning_rate`: smaller steps need more trees but often generalize better.
- `subsample` / `colsample_bytree`: row/column dropout against overfit.
- `min_child_weight`: ignore ultra-small child nodes.

Search spaces stay small on purpose: a portfolio project should finish on a laptop.


In [ ]:
import json
meta_path = ROOT / "models" / "model_metadata.json"
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    print("Selected model:", meta.get("model_name"))
    print("Threshold:", meta.get("threshold"))
    print("Best params:", meta.get("best_params"))
else:
    print("No metadata yet — train first.")
